In [0]:
storage_account = "your_storage_account_name"
storage_key = "your_storage_account_key"


spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
df = spark.read.parquet(
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/AAPL_20260726.parquet"
)

df.show(5)

+------------------+------------------+------------------+------------------+---------+-------------------+
|             Close|              High|               Low|              Open|   Volume|               Date|
+------------------+------------------+------------------+------------------+---------+-------------------+
| 145.2283172607422|146.04710543551434| 143.9708787526298| 144.5264945756414| 72434100|2021-07-26 00:00:00|
|143.06434631347656| 145.4427434254304|141.87514775749966|145.35500444699645|104818600|2021-07-27 00:00:00|
|141.31956481933594|143.25932698356266|138.94116715524945|141.15385873599058|118931200|2021-07-28 00:00:00|
|141.96286010742188| 142.8498878684949|140.92962551695572|141.03684881635502| 56699500|2021-07-29 00:00:00|
| 142.1772918701172|142.63542638578699| 140.4714762953773|140.73466343435456| 70440600|2021-07-30 00:00:00|
+------------------+------------------+------------------+------------------+---------+-------------------+
only showing top 5 rows


In [0]:
df.printSchema()
print("Rows:", df.count())

root
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Date: timestamp_ntz (nullable = true)

Rows: 1255


In [0]:
from pyspark.sql.functions import col

silver_df = (
    df.dropDuplicates()
      .dropna()
)

print("Rows after cleaning:", silver_df.count())

silver_df.show(5)

Rows after cleaning: 1255
+------------------+------------------+------------------+------------------+--------+-------------------+
|             Close|              High|               Low|              Open|  Volume|               Date|
+------------------+------------------+------------------+------------------+--------+-------------------+
|171.36322021484375|174.53480585728352| 170.7367358216346|173.75169663163055|73401800|2022-04-05 00:00:00|
|149.54954528808594| 149.6968066869272|146.62413491177585|147.34075911907576|74732300|2022-10-25 00:00:00|
|165.38111877441406|166.96523521407434| 164.8959946340493|166.36129252101995|43122900|2024-04-18 00:00:00|
|192.80978393554688|194.81244022862765|192.50244866402238| 194.0093990023229|41181800|2024-06-06 00:00:00|
|230.99417114257812|231.19289841511937|226.82083777153602| 227.0791817096477|39620300|2025-02-05 00:00:00|
+------------------+------------------+------------------+------------------+--------+-------------------+
only showin

In [0]:
from pyspark.sql.functions import count, when

silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()


+-----+----+---+----+------+----+
|Close|High|Low|Open|Volume|Date|
+-----+----+---+----+------+----+
|    0|   0|  0|   0|     0|   0|
+-----+----+---+----+------+----+



In [0]:
silver_df.write.mode("overwrite").parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/AAPL_clean.parquet"
)

print("✅ Silver Layer created successfully.")
silver_df.count()

✅ Silver Layer created successfully.


1255